# 02 — White matter lesion segmentation with SAMSEG

**SAMSEG** (Sequence Adaptive Multimodal SEGmentation) is part of FreeSurfer 7+ and segments brain structures *and* WM lesions from any combination of T1 and FLAIR (or other) contrasts — without retraining.

Why SAMSEG for MS?
- Sequence-agnostic (handles different scanner protocols)
- Joint structural + lesion segmentation
- Open-source, reproducible

Inputs assumed: preprocessed `T1_brain.nii.gz` and `FLAIR_in_T1.nii.gz` from notebook 01.

In [ ]:
from pathlib import Path
ANAT = Path('/neurodesktop-storage/ms-workshop-out/sub-001/anat')
OUT  = Path('/neurodesktop-storage/ms-workshop-out/sub-001/samseg')
OUT.mkdir(parents=True, exist_ok=True)
print(list(ANAT.iterdir()))

## Run SAMSEG with lesion segmentation

In [ ]:
%%bash -s "$ANAT" "$OUT"
ml load freesurfer
ANAT=$1; OUT=$2
run_samseg \
  --input  $ANAT/T1_brain.nii.gz $ANAT/FLAIR_in_T1.nii.gz \
  --pallidum-separate \
  --lesion --lesion-mask-pattern 0 1 \
  --threads 4 \
  --output $OUT

`--lesion-mask-pattern 0 1` tells SAMSEG that lesions are **dark in T1 (input 0) and bright in FLAIR (input 1)** — the classic MS lesion appearance.

Outputs:
- `seg.mgz` — full segmentation including lesion label (99)
- `samseg.stats` — per-structure volumes, including total lesion volume

## Extract and report lesion volume

In [ ]:
import re
stats = (OUT / 'samseg.stats').read_text()
for line in stats.splitlines():
    if 'Lesions' in line or 'lesion' in line.lower():
        print(line)

## Visualise the lesion mask overlaid on FLAIR

In [ ]:
%%bash -s "$OUT" "$ANAT"
ml load freesurfer
OUT=$1; ANAT=$2
# Extract lesion-only mask (label 99) from seg.mgz
mri_binarize --i $OUT/seg.mgz --match 99 --o $OUT/lesions.nii.gz

In [ ]:
from nilearn import plotting
plotting.plot_roi(
    str(OUT/'lesions.nii.gz'),
    bg_img=str(ANAT/'FLAIR_in_T1.nii.gz'),
    title='SAMSEG WM lesion mask on FLAIR',
    cmap='autumn'
)

## Clinical interpretation

- **Total lesion volume** is a coarse but useful biomarker (correlates weakly with disability — the *clinico-radiological paradox*).
- **Lesion location** (periventricular, juxtacortical, infratentorial, spinal) carries diagnostic weight per McDonald 2017 criteria.
- For longitudinal studies, run SAMSEG separately at each timepoint or use the longitudinal pipeline to constrain segmentation.

## Alternatives to consider

- **LST** (Lesion Segmentation Toolbox, SPM): widely used, lesion-prediction algorithm (LPA) and lesion-growth algorithm (LGA)
- **nicMSlesions**, **DeepLesionBrain**: deep-learning approaches (require GPU)
- **BIANCA** (FSL): supervised, needs training data